# Building a modified protein with mBuild that can be read by OpenFF
### Joseph R. Laforet Jr.

Semaglutide (Ozempic) is a 31-residue peptide with a lipid linker attached to Lysine 20. The OpenFF post-translational-modification [workshop](https://github.com/openforcefield/2026-virtual-workshops/blob/main/ptm/ptm-workshop.ipynb) simulates
it, starting from a prepared PDB file that someone had to get from somewhere...

This notebook is an example of "somewhere". We begin with the unmodified peptide,
attach the linker with mBuild, and then read the result back with
openff-pablo using the OpenFF workshop's loader.

## Building the starting structure:

Let's start with an ordinary protonated peptide. This comes from somewhere ELSE, left as an exercise for the reader. TODO: Consider building the peptide as a `mb.Polymer`. Residue 2 is AIB, and residue 20 is the
lysine that will carry the linker — protonated, as it is at neutral pH.

AIB is not one of the twenty standard residues, so its template is
downloaded from the RCSB on first use.

In [ ]:
from mbuild.biopolymers import Protein

mbuild_protein = Protein("semaglutide_apo.pdb", download=True)
print(mbuild_protein.n_particles, "atoms, net formal charge", mbuild_protein.net_formal_charge)

mbuild_lysine = mbuild_protein.get_residue(20, chain_id="A")
print("residue 2:", mbuild_protein.get_residue(2, chain_id="A").name)
print(f"LYS 20: charge {mbuild_lysine.formal_charge:+d}")

In [ ]:
mbuild_protein.visualize()

## Adding the modification:

The OpenFF workshop models
semaglutide's lipid linker as a residue named `KUT`, bonded to lysine 20.
That three-letter name is its code in the wwPDB Chemical Component
Dictionary ([KUT at RCSB](https://www.rcsb.org/ligand/KUT)). The cell
below reads the non-standard residue names straight out of the reference
file. It finds two: `AIB`, the alpha-aminoisobutyric acid at position 2,
which is an ordinary residue the loader already matched, and `KUT`.

You *could* build the `KUT` modification from SMILES instead, but then its atoms would
get names that aren't too useful, and a CCD-based residue library would need a
hand-written definition to recognise it. If we built the fragment from a CCD component, the
atoms already carry the names the library expects.

`link_atom` is the atom of the fragment that forms the bond to the protein.
Here that is `C33`, the carbon that bonds to the lysine `NZ`. The
OpenFF workshop's own loader call declares the same pair, `linking_atoms=["NZ",
"C33"]`, which is where the name is taken from.


In [ ]:
from mbuild.biopolymers import fragment_from_ccd

residue_names = {line[17:20] for line in open("semaglutide_reference.pdb") if line.startswith("HETATM")}
print("non-standard residues in the workshop structure:", sorted(residue_names))

mbuild_linker = fragment_from_ccd("KUT", link_atom="C33")
print(mbuild_linker.n_particles, "atoms, formal charge", mbuild_linker.formal_charge)
print("bonds to the protein through:", mbuild_linker.link_atoms["1"])


## Making the bond between the peptide and the fragment:

We need two function calls:

1.  `deprotonate` takes the proton off the lysine side chain,
because the ammonium ion at neutral pH has no lone pair and does participate in acylation; the neutral amine does.

2.  `attach` then removes one hydrogen
from each side and forms the bond.

`leaving_atom_names` says which hydrogen is removed. The three hydrogens on
that nitrogen are chemically equivalent, so the choice is arbitrary from a chemistry perspective, but the residue library downstream describes the product by
naming the atom that is *absent*, so the file has to agree with it.

We then rigidly place the linker. It is aligned along the bond the linker replaces,
using its CCD geometry as an initial conformer, so it may land in a neighbouring side chain. When any
of its atoms comes within 2 Å of an existing atom, `attach` relaxes the
fragment with the protein held fixed, which takes a second or two here.
`relax=True` is the default and is written out only to make that visible.


In [ ]:
mbuild_protein.deprotonate(20, "NZ", chain_id="A")
bond_record = mbuild_protein.attach(
    mbuild_linker,
    resnum=20,
    atom_name="NZ",
    chain_id="A",
    leaving_atom_names="HZ2",
    relax=True,
)

print(f"{bond_record.residue1.name} {bond_record.atom1_name} - "
      f"{bond_record.residue2.name} {bond_record.atom2_name}")
print("hydrogens removed:", bond_record.leaving1 + bond_record.leaving2)
print(mbuild_protein.n_particles, "atoms, net formal charge", mbuild_protein.net_formal_charge)

Look at the product before writing anything. `visualize` draws the
protein as a cartoon and the attached residue as licorice, from the
same PDB text that `save_pdb` writes.

In [ ]:
view = mbuild_protein.visualize()
view.center(selection="[KUT]")
view

In [ ]:
mbuild_protein.save_pdb("semaglutide_mbuild.pdb", overwrite=True)
print(mbuild_protein.bond_records()[-1])

## Reading the modified protein into OpenFF via Pablo:

This is the OpenFF workshop's loader call, copied unchanged. It declares the
crosslink: which two residues, which two atoms bond, and which atom
leaves each side. Those are the same names `attach` was given.

Nothing else is needed — no hand-written residue definition, no sidecar
file. `KUT` is a CCD component, so pablo already knows it.

In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

STD_CCD_CACHE.auto_download = True
pablo_residue_library = STD_CCD_CACHE.with_crosslink(
    residues=["LYS", "KUT"],
    linking_atoms=["NZ", "C33"],
    leaving_atoms=[["HZ2"], ["H61"]],
    bond_order=1,
)

openff_topology = topology_from_pdb("semaglutide_mbuild.pdb", residue_library=pablo_residue_library)
openff_molecule = openff_topology.molecule(0)
print(openff_topology.n_molecules, "molecule:", openff_molecule.n_atoms, "atoms,",
      openff_molecule.hill_formula + ",", "net charge", openff_molecule.total_charge)

Notice we have one molecule. The linker is part of the peptide!

In [ ]:
view = openff_topology.visualize()
view.clear_representations()
view.add_representation("cartoon", color="#990000")
view.add_representation("licorice", selection="not [KUT]", color="#990000")
view.add_representation("licorice", selection="[KUT] or 20:A", color="#FF7733")
view.center(selection="[KUT]")
view

In [ ]:
openff_molecule.visualize("rdkit", show_all_hydrogens=False)

## Did we build the right molecule?

`semaglutide_reference.pdb` is chain A of the OpenFF workshop's `7KI0_prepared.pdb`. 
Reading it through the same loader call, with the
same crosslink declaration, allows us to compare the two
OpenFF molecules.

In [ ]:
from collections import Counter

openff_reference_topology = topology_from_pdb("semaglutide_reference.pdb", residue_library=pablo_residue_library)
openff_reference_molecule = openff_reference_topology.molecule(0)


def atoms_by_element_and_charge(molecule):
    return Counter((atom.atomic_number, atom.formal_charge.m) for atom in molecule.atoms)


def bonds_by_elements_and_order(molecule):
    return Counter(
        (tuple(sorted((bond.atom1.atomic_number, bond.atom2.atomic_number))), bond.bond_order)
        for bond in molecule.bonds
    )


checks = {
    "one molecule, so the linker is bonded": openff_topology.n_molecules == 1,
    "same atom count": openff_molecule.n_atoms == openff_reference_molecule.n_atoms,
    "same formula": openff_molecule.hill_formula == openff_reference_molecule.hill_formula,
    "same net charge": openff_molecule.total_charge == openff_reference_molecule.total_charge,
    "same atoms, by element and formal charge":
        atoms_by_element_and_charge(openff_molecule) == atoms_by_element_and_charge(openff_reference_molecule),
    "same bonds, by elements and order":
        bonds_by_elements_and_order(openff_molecule) == bonds_by_elements_and_order(openff_reference_molecule),
    "isomorphic, setting stereochemistry aside": openff_molecule.is_isomorphic_with(
        openff_reference_molecule, atom_stereochemistry_matching=False, bond_stereochemistry_matching=False
    ),
}
for label, passed in checks.items():
    print("PASS" if passed else "FAIL", label)


The last check intentionally disregards stereochemistry. The next cell puts
it back and lists every stereocenter where the two molecules disagree,
by residue and atom name.


In [ ]:
def stereo_by_site(molecule):
    return {
        (atom.metadata["residue_name"], atom.metadata["canonical_name"]): atom.stereochemistry
        for atom in molecule.atoms
        if atom.stereochemistry
    }


built, reference = stereo_by_site(openff_molecule), stereo_by_site(openff_reference_molecule)
print(len(built), "stereocenters in the built molecule,", len(reference), "in the reference")
for site in sorted(set(built) | set(reference)):
    if built.get(site) != reference.get(site):
        print(f"differs at {site[0]} {site[1]}: built {built.get(site)}, deposited {reference.get(site)}")


Besides that one stereocenter, the two are the same molecule!

mBuild placed the linker at the geometry its CCD component defines; at that stereocenter the OpenFF deposited
coordinates disagree with the CCD. They are two different enantiomers, and a loader that reads chemistry
rather than coordinates is what tells you so. I make no claims as to which stereocenter is correct, since I don't know where the OpenFF semaglutide came from. We are simply showcasing that we can match templates from the CCD.

## Assigning force field parameters and running a short MD simulation

We follow usual simulation protocols from here.

In [ ]:
from openff.toolkit import ForceField

openff_force_field = ForceField("openff_no_water-3.0.0-alpha0.offxml", "opc3.offxml")
openff_interchange = openff_force_field.create_interchange(openff_topology)
print("parameterized", openff_interchange.topology.n_atoms, "atoms")

In [ ]:
import openmm
from openmm import unit

openmm_simulation = openff_interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picosecond, 2.0 * unit.femtosecond
    ),
)
openmm_simulation.minimizeEnergy(maxIterations=200)
openmm_simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
openmm_simulation.step(500)
openmm_state = openmm_simulation.context.getState(getEnergy=True)
print("potential energy:", openmm_state.getPotentialEnergy())